# CFAR-YOLOv11 — Resource-Optimised Dark-Vessel Detection in SAR Imagery

End-to-end experiment notebook for the proposal *"Resource Optimised Detection of Dark Vessels ... based on Robust CFAR algorithm and Lightweight Deep Learning Models"*, implemented on **YOLOv11**.

**Novel model** (`yolo11-cfar.yaml`, code in `ultralytics/nn/modules/cfar.py`):
1. **RobustCFARGate** — a differentiable, *parameterised* robust CFAR (two-pass truncated-statistics clutter estimation over a guard-band annulus, learnable per-channel detection threshold) embedded as attention at P2 (backbone) and P3 (head). 3 params/channel, zero convolutions.
2. **C3k2DCA** — Dense Context-Aware redesign of YOLOv11's C3k2 (successor of the proposal's modified C2f): multi-dilation depthwise pyramid + global-context channel gate, densely aggregated via the CSP topology.

See `research/README.md` in the repo for the full design document and proposal mapping.

> **Runtime → Change runtime type → GPU (T4 or better)** before running.

In [ ]:
!nvidia-smi

## 1. Setup — clone the custom repo (branch `yolov11-CFAR`) and install

In [ ]:
%cd /content
import os
if not os.path.isdir('/content/Experiment5'):
    !git clone -b yolov11-CFAR https://github.com/sri2498/Experiment5.git
%cd /content/Experiment5
%pip install -qe .

import ultralytics
from ultralytics.nn.modules import RobustCFARGate, C3k2DCA  # verify novel modules are importable
print('ultralytics', ultralytics.__version__, '| CFAR modules OK')

## 2. Sanity check — build CFAR-YOLOv11 and run a dummy forward pass
This must print the layer table including `RobustCFARGate` and `C3k2DCA` and complete a forward pass before any training is attempted.

In [ ]:
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo11n-cfar.yaml')  # scale letter (n/s/m/l/x) selects compound scaling
model.info(detailed=False)
_ = model.predict(np.zeros((640, 640, 3), dtype=np.uint8), verbose=False)  # smoke test
print('forward pass OK')

baseline = YOLO('yolo11n.yaml')
baseline.info(detailed=False)  # compare parameter/GFLOP budget with the baseline

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. HRSID dataset preparation
Your Drive HRSID folder already contains YOLO-format splits (`Train/`, `validation/`, `test/`) plus an old `google_colab_config.yaml` and stale `*.cache` files.
This cell: **(a)** locates the folder (set `HRSID_DRIVE` manually if auto-search fails), **(b)** copies it to fast local disk, **(c)** removes stale caches, **(d)** writes a fresh `hrsid.yaml` with correct absolute paths.

> If your folder is under *Shared with me*, first add a shortcut to *My Drive* (right-click → Organise → Add shortcut).

In [ ]:
import glob, os, shutil, yaml
from pathlib import Path

HRSID_DRIVE = ''  # <-- optionally set manually, e.g. '/content/drive/MyDrive/HRSID'

if not HRSID_DRIVE:
    cands = [os.path.dirname(p) for p in glob.glob('/content/drive/MyDrive/**/google_colab_config.yaml', recursive=True)]
    cands += [d.rstrip('/') for d in glob.glob('/content/drive/MyDrive/**/', recursive=True)
              if os.path.isdir(os.path.join(d, 'Train')) and os.path.isdir(os.path.join(d, 'validation'))]
    assert cands, 'HRSID folder not found under MyDrive — set HRSID_DRIVE manually.'
    HRSID_DRIVE = cands[0]
print('HRSID source:', HRSID_DRIVE)

LOCAL = Path('/content/datasets/hrsid')
if not LOCAL.exists():
    ignore = shutil.ignore_patterns('*.cache', '.ipynb_checkpoints', 'yolov8_improved_exp*', '*.pt')
    shutil.copytree(HRSID_DRIVE, LOCAL, ignore=ignore)
for c in LOCAL.rglob('*.cache'):
    c.unlink()  # stale label caches from previous runs would poison training

# class names: reuse the old config if readable, else single class 'ship'
names = {0: 'ship'}
old_cfg = LOCAL / 'google_colab_config.yaml'
if old_cfg.exists():
    try:
        d = yaml.safe_load(old_cfg.read_text()) or {}
        n = d.get('names')
        if isinstance(n, list):
            names = {i: v for i, v in enumerate(n)}
        elif isinstance(n, dict):
            names = n
    except Exception as e:
        print('could not parse old config, defaulting to ship:', e)

splits = {s: next((LOCAL / c for c in (s, s.capitalize(), s.lower()) if (LOCAL / c).is_dir()), None)
          for s in ('Train', 'validation', 'test')}
assert splits['Train'] and splits['validation'], f'missing split dirs in {LOCAL}: {list(LOCAL.iterdir())}'

DATA_HRSID = '/content/datasets/hrsid/hrsid.yaml'
cfg = {'path': str(LOCAL), 'train': splits['Train'].name, 'val': splits['validation'].name, 'names': names}
if splits['test']:
    cfg['test'] = splits['test'].name
Path(DATA_HRSID).write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path(DATA_HRSID).read_text())

### 4.1 Dataset analysis (sanity + paper statistics)
Counts, box-size distribution (ships in HRSID are predominantly very small — the regime the CFAR gate and DCA context target), and a sample grid.

In [ ]:
import matplotlib.pyplot as plt
import cv2, random

def split_stats(d):
    imgs = sorted([p for e in ('*.jpg','*.jpeg','*.png','*.tif','*.tiff','*.bmp') for p in Path(d).rglob(e)])
    n_box, areas = 0, []
    for im in imgs:
        t = im.with_suffix('.txt')
        if not t.exists():
            t = Path(str(im.parent).replace(os.sep+'images', os.sep+'labels')) / (im.stem + '.txt')
        if t.exists():
            for line in t.read_text().split('\n'):
                if line.strip():
                    _, _, _, w, h = map(float, line.split()[:5])
                    areas.append((w * h) ** 0.5)  # sqrt of normalised area
                    n_box += 1
    return imgs, n_box, areas

all_areas = {}
for s, d in splits.items():
    if d:
        imgs, nb, areas = split_stats(d)
        all_areas[s] = (imgs, areas)
        print(f'{s:>12}: {len(imgs)} images, {nb} boxes')

imgs, areas = all_areas['Train']
plt.figure(figsize=(5, 3))
plt.hist([a * 800 for a in areas], bins=60)
plt.xlabel('sqrt(box area) [px @ 800]'); plt.ylabel('count'); plt.title('HRSID ship sizes'); plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, im in zip(axes.flat, random.sample(imgs, 8)):
    img = cv2.imread(str(im)); H, W = img.shape[:2]
    t = im.with_suffix('.txt')
    if t.exists():
        for line in t.read_text().split('\n'):
            if line.strip():
                _, cx, cy, w, h = map(float, line.split()[:5])
                x1, y1 = int((cx - w/2) * W), int((cy - h/2) * H)
                x2, y2 = int((cx + w/2) * W), int((cy + h/2) * H)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(img[..., ::-1]); ax.axis('off')
plt.suptitle('HRSID samples with ground truth'); plt.show()

### 4.2 Classical robust CFAR on a real chip (analysis figure)
The image-domain counterpart of the in-network `RobustCFARGate` — useful as a paper figure motivating the design.

In [ ]:
import sys
sys.path.insert(0, '/content/Experiment5/research')
from cfar_demo import robust_cfar

chip = str(random.choice(all_areas['Train'][0]))
g = cv2.imread(chip, cv2.IMREAD_GRAYSCALE)
z, mask = robust_cfar(g, k_bg=41, k_guard=21, t_trunc=2.0, tau=5.0)
overlay = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB); overlay[mask > 0] = (255, 64, 64)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, ttl, kw in zip(axes, [g, np.clip(z, 0, 10), overlay],
                           ['SAR chip', 'robust CFAR statistic z', 'candidates (z ≥ 5)'],
                           [{'cmap': 'gray'}, {'cmap': 'inferno'}, {}]):
    ax.imshow(im, **kw); ax.set_title(ttl); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Training — baseline YOLOv11n vs CFAR-YOLOv11n
Both models are trained **from scratch** (random init) with identical hyperparameters and seed, so the comparison isolates the architectural contribution. Adjust `EPOCHS`/`BATCH` to your GPU quota; 150 epochs at 800 px on a T4 takes roughly 6–8 h per model — for a quick first signal use 50 epochs.

> Colab sessions can disconnect: `resume=True` on a rerun continues an interrupted training (see §7 to persist runs to Drive).

In [ ]:
EPOCHS = 150   # quick signal: 50
IMGSZ  = 800   # HRSID native chip size
BATCH  = 16    # reduce if OOM
SEED   = 0
PROJECT = '/content/runs/hrsid'
RUN_BASELINE = True

common = dict(data=DATA_HRSID, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
              project=PROJECT, exist_ok=True, patience=50, workers=8, plots=True)

In [ ]:
if RUN_BASELINE:
    YOLO('yolo11n.yaml').train(name='baseline_yolo11n', **common)

In [ ]:
YOLO('yolo11n-cfar.yaml').train(name='cfar_yolo11n', **common)

## 6. Evaluation and comparison

In [ ]:
import pandas as pd

rows = []
for name in (['baseline_yolo11n'] if RUN_BASELINE else []) + ['cfar_yolo11n']:
    w = f'{PROJECT}/{name}/weights/best.pt'
    m = YOLO(w)
    n_l, n_p, n_g, flops = m.info(verbose=False)
    for split in ['val'] + (['test'] if 'test' in yaml.safe_load(Path(DATA_HRSID).read_text()) else []):
        r = m.val(data=DATA_HRSID, split=split, imgsz=IMGSZ, plots=(split != 'val'), verbose=False)
        rows.append({'model': name, 'split': split,
                     'precision': round(r.results_dict['metrics/precision(B)'], 4),
                     'recall': round(r.results_dict['metrics/recall(B)'], 4),
                     'mAP50': round(r.results_dict['metrics/mAP50(B)'], 4),
                     'mAP50-95': round(r.results_dict['metrics/mAP50-95(B)'], 4),
                     'params(M)': round(n_p / 1e6, 3), 'GFLOPs': round(flops, 2)})
df = pd.DataFrame(rows)
display(df)
df.to_csv('/content/runs/hrsid/comparison.csv', index=False)

In [ ]:
# Qualitative side-by-side on random test/val chips
src = splits['test'] or splits['validation']
samples = random.sample(sorted([p for e in ('*.jpg','*.png') for p in Path(src).rglob(e)]), 4)
models = {n: YOLO(f'{PROJECT}/{n}/weights/best.pt') for n in
          ((['baseline_yolo11n'] if RUN_BASELINE else []) + ['cfar_yolo11n'])}
fig, axes = plt.subplots(len(samples), len(models), figsize=(7 * len(models), 6 * len(samples)))
axes = np.atleast_2d(axes)
for i, s in enumerate(samples):
    for j, (n, m) in enumerate(models.items()):
        r = m.predict(str(s), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
        axes[i, j].imshow(r.plot()[..., ::-1]); axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(n)
plt.tight_layout(); plt.show()

## 7. Persist results to Drive

In [ ]:
DEST = '/content/drive/MyDrive/CFAR_YOLOv11_runs'
!mkdir -p {DEST}
!cp -r /content/runs/hrsid {DEST}/
print('saved to', DEST)

## 8. Custom dataset (Sentinel-1A VH chips)
Your custom folder holds Sentinel-1A GRD chips (VH polarisation) with YOLO `.txt` labels of matching stems. This cell pairs images↔labels recursively, makes an 80/20 train/val split, writes `custom.yaml`, and (optionally) **fine-tunes the HRSID-trained CFAR model** — the recommended transfer strategy for a small custom set.

In [ ]:
CUSTOM_DRIVE = ''  # <-- set to your custom dataset folder, e.g. '/content/drive/MyDrive/S1A_custom'
assert CUSTOM_DRIVE, 'set CUSTOM_DRIVE to your custom dataset folder on Drive'

IMG_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
srcs = sorted(Path(CUSTOM_DRIVE).rglob('*'))
imgs = {p.stem: p for p in srcs if p.suffix.lower() in IMG_EXT}
lbls = {p.stem: p for p in srcs if p.suffix.lower() == '.txt'}
pairs = [(imgs[s], lbls.get(s)) for s in sorted(imgs)]
print(f'{len(imgs)} images, {len(lbls)} label files, {sum(1 for _, l in pairs if l)} matched pairs')
assert pairs, 'no images found — check CUSTOM_DRIVE'

CROOT = Path('/content/datasets/custom')
random.Random(SEED).shuffle(pairs)
n_val = max(1, int(0.2 * len(pairs)))
for sub, chunk in {'val': pairs[:n_val], 'train': pairs[n_val:]}.items():
    (CROOT / 'images' / sub).mkdir(parents=True, exist_ok=True)
    (CROOT / 'labels' / sub).mkdir(parents=True, exist_ok=True)
    for im, lb in chunk:
        shutil.copy2(im, CROOT / 'images' / sub / im.name)
        (CROOT / 'labels' / sub / (im.stem + '.txt')).write_text(lb.read_text() if lb else '')

DATA_CUSTOM = str(CROOT / 'custom.yaml')
Path(DATA_CUSTOM).write_text(yaml.safe_dump({'path': str(CROOT), 'train': 'images/train',
                                             'val': 'images/val', 'names': {0: 'ship'}}, sort_keys=False))
print(Path(DATA_CUSTOM).read_text())

In [ ]:
FINETUNE_FROM_HRSID = True
start = f'{PROJECT}/cfar_yolo11n/weights/best.pt' if FINETUNE_FROM_HRSID else 'yolo11n-cfar.yaml'
m = YOLO(start)
m.train(data=DATA_CUSTOM, epochs=100, imgsz=800, batch=16, seed=SEED,
        project='/content/runs/custom', name='cfar_yolo11n_custom', exist_ok=True, plots=True)
m.val(data=DATA_CUSTOM, imgsz=800)
!mkdir -p {DEST}
!cp -r /content/runs/custom {DEST}/

## 9. Ablations (optional)
- **Gates only**: in `ultralytics/cfg/models/11/yolo11-cfar.yaml` replace `C3k2DCA` with `C3k2`.
- **DCA only**: remove the two `RobustCFARGate` lines and renumber the affected `Concat`/`Detect` indices.
- **Gate windows**: `RobustCFARGate` args are `[k_bg, k_guard]` (optional third: truncation depth `t_trunc`).

Report `comparison.csv` plus the per-run `results.png`, PR curves and confusion matrices saved under each run directory.

---
*Model: CFAR-YOLOv11 — differentiable truncated-statistics CFAR gating + Dense Context-Aware C3k2, built on Ultralytics YOLOv11 (AGPL-3.0).*